# LLM Prompt Autoencoder (Encode → Decode → Refine)

This notebook implements a prompt autoencoder loop using an LLM for both encoding and decoding. It iteratively refines the system prompts to maximize mutual information between the original and decoded prompt.

**Workflow**
1. Encode the input prompt using an encoder system prompt.
2. Decode the encoded representation using a decoder system prompt.
3. Estimate mutual information between original and decoded text.
4. Log results and refine the system prompts using the LLM.
5. Repeat until the score converges.

In [ ]:
# --- Setup ---
import json
import re
import zlib
from dataclasses import dataclass
from typing import List, Tuple

import pandas as pd

try:
    import ollama
except Exception as e:
    raise RuntimeError("ollama is required. Install and ensure 'ollama serve' is running.") from e

MODEL_NAME = "llama3.1:8b"  # change if needed

# Sample prompt to encode (replace with your own transcript or text)
ORIGINAL_PROMPT = """
Meeting transcript:
- Alex: We need to reduce onboarding time by 30% this quarter.
- Priya: Let's focus on reducing steps in the signup flow.
- Ming: We'll also add tooltips and a quickstart checklist.
- Alex: Metrics should include activation rate and time-to-first-value.
""".strip()

ENCODER_SYSTEM_PROMPT = """
You are an LLM encoder. Convert the user's prompt into a structured representation.
Rules:
- Preserve all facts, constraints, and key entities.
- Use a JSON schema with keys: title, entities, actions, constraints, metrics, timeline, notes.
- Be concise and lossless.
- Output ONLY valid JSON.
""".strip()

DECODER_SYSTEM_PROMPT = """
You are an LLM decoder. Reconstruct the original prompt from the structured JSON.
Rules:
- Use all information in the JSON.
- Preserve meaning and detail as much as possible.
- Output plain text only (no JSON, no markdown).
""".strip()

@dataclass
class IterationResult:
    iteration: int
    encoded: str
    decoded: str
    mi_bits: float
    encoder_system_prompt: str
    decoder_system_prompt: str

def ollama_chat(system_prompt: str, user_prompt: str, model: str = MODEL_NAME) -> str:
    client = ollama.Client()
    resp = client.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        options={"temperature": 0.3}
    )
    return resp["message"]["content"]

def compression_entropy_bits(text: str) -> float:
    """Estimate entropy using compressed length (bits)."""
    if not text:
        return 0.0
    data = text.encode("utf-8")
    compressed = zlib.compress(data, level=9)
    return len(compressed) * 8.0

def mutual_information_proxy(x: str, y: str) -> float:
    """Compression-based proxy for MI: I(X;Y) ≈ H(X) + H(Y) - H(X,Y)."""
    if not x and not y:
        return 0.0
    h_x = compression_entropy_bits(x)
    h_y = compression_entropy_bits(y)
    h_xy = compression_entropy_bits(x + "\n\n<SEP>\n\n" + y)
    return max(0.0, h_x + h_y - h_xy)

def normalize_json(text: str) -> str:
    """Attempt to extract and normalize JSON from LLM output."""
    match = re.search(r"\{.*\}", text, flags=re.S)
    if not match:
        return text.strip()
    raw = match.group(0)
    try:
        obj = json.loads(raw)
        return json.dumps(obj, ensure_ascii=False, sort_keys=True)
    except Exception:
        return raw.strip()

def refine_prompts(
    encoder_prompt: str,
    decoder_prompt: str,
    original: str,
    decoded: str,
    mi_bits: float,
    model: str = MODEL_NAME,
    ) -> Tuple[str, str]:
    system = (
        "You are a prompt engineer. Improve encoder/decoder system prompts\n"
        "to maximize mutual information between original and decoded text.\n"
        "Return JSON with keys: encoder_system_prompt, decoder_system_prompt.\n"
        "Keep prompts concise and enforce lossless reconstruction.\n"
    )
    user = (
        f"Current encoder prompt:\n{encoder_prompt}\n\n"
        f"Current decoder prompt:\n{decoder_prompt}\n\n"
        f"Original text:\n{original}\n\n"
        f"Decoded text:\n{decoded}\n\n"
        f"MI proxy (bits): {mi_bits:.2f}\n\n"
        "Provide improved prompts."
    )
    response = ollama_chat(system, user, model=model)
    response = normalize_json(response)
    try:
        obj = json.loads(response)
        return obj["encoder_system_prompt"], obj["decoder_system_prompt"]
    except Exception:
        return encoder_prompt, decoder_prompt

In [ ]:
# --- Single iteration runner ---
def run_iteration(
    iteration: int,
    encoder_prompt: str,
    decoder_prompt: str,
    original: str,
    refine: bool = True,
    model: str = MODEL_NAME,
    ) -> Tuple[IterationResult, str, str]:
    encoded = ollama_chat(encoder_prompt, original, model=model)
    encoded = normalize_json(encoded)
    decoded = ollama_chat(decoder_prompt, encoded, model=model)
    mi_bits = mutual_information_proxy(original, decoded)

    if refine:
        new_encoder, new_decoder = refine_prompts(
            encoder_prompt, decoder_prompt, original, decoded, mi_bits, model=model
        )
    else:
        new_encoder, new_decoder = encoder_prompt, decoder_prompt

    result = IterationResult(
        iteration=iteration,
        encoded=encoded,
        decoded=decoded,
        mi_bits=mi_bits,
        encoder_system_prompt=encoder_prompt,
        decoder_system_prompt=decoder_prompt,
    )
    return result, new_encoder, new_decoder

In [ ]:
# --- Run the training-like loop ---
NUM_ITERATIONS = 3  # increase as needed
results: List[IterationResult] = []
encoder_prompt = ENCODER_SYSTEM_PROMPT
decoder_prompt = DECODER_SYSTEM_PROMPT

for i in range(1, NUM_ITERATIONS + 1):
    result, encoder_prompt, decoder_prompt = run_iteration(
        i, encoder_prompt, decoder_prompt, ORIGINAL_PROMPT, refine=True
    )
    results.append(result)

df = pd.DataFrame([
    {
        "iteration": r.iteration,
        "mi_bits": r.mi_bits,
        "encoded": r.encoded,
        "decoded": r.decoded,
    }
    for r in results
],)
df

## Notes & Next Steps
- Replace `ORIGINAL_PROMPT` with your real transcript or complex text.
- If `refine=True`, the LLM will update system prompts each iteration.
- Consider swapping the MI proxy with a more rigorous estimator (e.g., token-level probabilities).
- Log `encoder_prompt` and `decoder_prompt` across iterations for auditing.